# VeriLens: real-time fact-checker (text + image)
Run cells in order. Restart the session after Cell 1.

### Cell 1: install, then Runtime > Restart session

In [ ]:
!pip install -q -U gradio openai google-genai tavily-python pillow
print("Installed. Now click Runtime > Restart session, then run Cell 2.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.6/58.6 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 59.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.5/64.5 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 68.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 103.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.9/267.9 kB 18.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.0 which is incompatible.
Installed. Now click Runtime > Restart session, then run Cell 2.


### Cell 2: keys and provider

In [ ]:
import os, getpass
from google.colab import userdata

PROVIDER = "gemini"         # changed from "openai"
os.environ["LLM_PROVIDER"] = PROVIDER

needed = ["TAVILY_API_KEY", "OPENAI_API_KEY" if PROVIDER == "openai" else "GEMINI_API_KEY"]
for name in needed:
    try:
        os.environ[name] = userdata.get(name)                     # from Colab Secrets
    except Exception:
        os.environ[name] = getpass.getpass(f"Paste {name}: ")       # or type it

print("Keys ready. Provider:", PROVIDER)

Keys ready. Provider: gemini


### Cell 3: the app (open the gradio.live link)

In [ ]:
import os, io, json, base64
from urllib.parse import urlparse
import gradio as gr
from tavily import TavilyClient


def require_api_keys():
    missing = []
    if "TAVILY_API_KEY" not in os.environ or not os.environ["TAVILY_API_KEY"]:
        missing.append("TAVILY_API_KEY")
    if os.environ.get("LLM_PROVIDER", "openai") == "openai":
        if "OPENAI_API_KEY" not in os.environ or not os.environ["OPENAI_API_KEY"]:
            missing.append("OPENAI_API_KEY")
    elif "GEMINI_API_KEY" not in os.environ or not os.environ["GEMINI_API_KEY"]:
        missing.append("GEMINI_API_KEY")
    if missing:
        raise RuntimeError("Missing API key(s): " + ", ".join(missing) + ". Add them in the notebook secrets or env before running the app.")


def safe_json_loads(raw):
    if raw is None:
        return {}
    if isinstance(raw, (dict, list)):
        return raw
    text = str(raw).strip()
    if not text:
        return {}
    if text.startswith("```"):
        text = text.strip("`\n ")
        if text.lower().startswith("json"):
            text = text[4:].strip()
    try:
        data = json.loads(text)
        return data if isinstance(data, (dict, list)) else {}
    except json.JSONDecodeError:
        try:
            start = text.find("{")
            end = text.rfind("}")
            if start != -1 and end != -1 and end > start:
                return json.loads(text[start:end + 1])
        except json.JSONDecodeError:
            pass
        return {}


def normalize_url(url):
    try:
        parsed = urlparse(url)
        return parsed.netloc.lower().replace("www.", "")
    except Exception:
        return ""


PROVIDER = os.environ.get("LLM_PROVIDER", "openai")
require_api_keys()
tavily = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])

if PROVIDER == "openai":
    from openai import OpenAI
    llm = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
    MODEL = "gpt-4o-mini"
else:
    from google import genai
    from google.genai import types
    llm = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
    MODEL = "gemini-2.5-flash"

# ---- Credible sources whitelist (edit freely) ----
TRUSTED = [
    "reuters.com", "apnews.com", "bbc.com", "bbc.co.uk", "aljazeera.com",
    "thehindu.com", "indianexpress.com", "ndtv.com", "hindustantimes.com",
    "snopes.com", "politifact.com", "factcheck.org", "fullfact.org",
    "altnews.in", "boomlive.in", "factly.in", "newschecker.in", "pib.gov.in",
    "who.int", "un.org", "mohfw.gov.in", "isro.gov.in", "nasa.gov",
    "nature.com", "science.org", "britannica.com",
]


def is_trusted_url(url):
    domain = normalize_url(url)
    return any(domain == trusted or domain.endswith("." + trusted) for trusted in TRUSTED)


EXTRACT_SYS = """You prepare content for fact-checking.
From the user's text and/or image, find the main verifiable factual claim.
If an image is given, read all text in it and note what it shows and what it claims.
Return JSON only: {"claim": "one self-contained sentence",
"queries": ["2 or 3 neutral web search queries to check it"],
"is_factual": true or false}
Set is_factual=false for pure opinion, jokes, predictions, or images with no factual claim."""

JUDGE_SYS = """You are a rigorous fact-checking analyst.
Use ONLY the numbered evidence provided. Never use outside memory.
Verdicts:
- Verified: supported by at least 2 independent reliable sources (copies of one wire story count as one)
- Misleading: partly true, missing context, outdated, misattributed, or contradicted by the evidence
- Unverified: not enough reliable evidence, or sources conflict. If unsure, choose Unverified.
Return JSON only: {"verdict": "Verified|Misleading|Unverified",
"confidence": 0.0-1.0, "rationale": "3-5 plain sentences",
"citations": [evidence numbers you relied on]}"""

ICON = {"Verified": "✅", "Misleading": "⚠️", "Unverified": "❓", "Not a factual claim": "💬"}


def pil_to_data_url(img):
    buf = io.BytesIO()
    img.convert("RGB").save(buf, format="JPEG", quality=90)
    return "data:image/jpeg;base64," + base64.b64encode(buf.getvalue()).decode()


def ask_json(system, text, image=None):
    if PROVIDER == "openai":
        content = [{"type": "text", "text": text}]
        if image is not None:
            content.append({"type": "image_url", "image_url": {"url": pil_to_data_url(image)}})
        resp = llm.chat.completions.create(
            model=MODEL, temperature=0.1,
            response_format={"type": "json_object"},
            messages=[{"role": "system", "content": system},
                      {"role": "user", "content": content}])
        raw = resp.choices[0].message.content
    else:
        parts = ([image] if image is not None else []) + [text]
        resp = llm.models.generate_content(
            model=MODEL, contents=parts,
            config=types.GenerateContentConfig(
                system_instruction=system,
                response_mime_type="application/json",
                temperature=0.1))
        raw = resp.text
    data = safe_json_loads(raw)
    if isinstance(data, list):
        return data[0] if data else {}
    return data if isinstance(data, dict) else {}


def search(queries, only_trusted):
    evidence, seen = [], set()
    for q in queries[:3]:
        kwargs = dict(query=q, search_depth="advanced", max_results=5)
        if only_trusted:
            kwargs["include_domains"] = TRUSTED
        try:
            res = tavily.search(**kwargs)
        except Exception as e:
            print("Search error:", e)
            continue
        for r in res.get("results", []):
            url = r.get("url", "")
            if not url or url in seen:
                continue
            if only_trusted and not is_trusted_url(url):
                continue
            seen.add(url)
            evidence.append({"title": r.get("title", "Untitled"), "url": url,
                             "text": (r.get("content") or "")[:1200]})
    return evidence


def verify(text, image, only_trusted):
    if not text and image is None:
        return "Please enter a statement or upload an image."
    try:
        user_text = "Content to fact-check:\n" + (text or "(see attached image)")
        ex = ask_json(EXTRACT_SYS, user_text, image)
        if not isinstance(ex, dict):
            ex = {}
        claim = (ex.get("claim") or text or "Claim from image").strip()
        if not claim:
            return "Please provide a claim or upload an image with a factual statement."
        if not ex.get("is_factual", True):
            return (f"## {ICON['Not a factual claim']} Not a factual claim\n\n"
                    f"**Detected:** {claim}\n\nThis looks like opinion, satire, a prediction, "
                    "or an image with no checkable claim.")

        evidence = search(ex.get("queries") or [claim], only_trusted)
        if not evidence:
            return (f"## {ICON['Unverified']} Unverified\n\n**Claim:** {claim}\n\n"
                    "No reliable reporting was found. For very recent news, untick "
                    "'Only trusted sources' and try again.")

        ev_text = "\n\n".join(f"[{i+1}] {e['title']} ({e['url']})\n{e['text']}"
                              for i, e in enumerate(evidence))
        j = ask_json(JUDGE_SYS, f"CLAIM: {claim}\n\nEVIDENCE:\n{ev_text}")
        if not isinstance(j, dict):
            j = {}

        cites = [evidence[n-1] for n in j.get("citations", [])
                 if isinstance(n, int) and 1 <= n <= len(evidence)]
        v = j.get("verdict", "Unverified")
        confidence = float(j.get("confidence", 0.0) or 0.0)
        out = (f"## {ICON.get(v, '?')} {v}  ?  confidence {confidence:.0%}\n\n"
               f"**Claim checked:** {claim}\n\n{j.get('rationale', '')}\n\n### Sources\n")
        out += "\n".join(f"- [{c['title']}]({c['url']})" for c in cites) or "_No citations returned._"
        return out
    except Exception as e:
        return f"Error: {e}"
demo = gr.Interface(
    fn=verify,
    inputs=[
        gr.Textbox(lines=5, label="Statement / news / tweet text"),
        gr.Image(type="pil", label="Image or screenshot (optional)"),
        gr.Checkbox(value=True, label="Only trusted sources"),
    ],
    outputs=gr.Markdown(label="Verdict"),
    title="VeriLens - real-time fact-checker",
    description="Checks your claim against live, credible news and fact-check sources.",
    flagging_mode="never",
)
demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://4ec318b0339f8521ff.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
